# REINVENT4 + MOST/UV — FINAL v5

**Один notebook. Никаких дополнительных файлов.**

Исправлено:
- безопасная распаковка symlink из upstream REINVENT4;
- отсутствующий SciPy;
- несовместимое `parameters.unique_sequences`, которое REINVENT 4.8.24 отклоняет;
- добавлена реальная Pydantic-проверка RL-конфига **до** старта обучения.

Zenodo больше не используется. `reinvent.prior` берётся из официальной истории GitHub REINVENT4 и проверяется по точному Git blob SHA. Если в текущем Colab уже есть установленная среда из предыдущего запуска, notebook **переиспользует её** и не скачивает всё заново.

Перед запуском включите GPU и нажмите **Runtime → Run all**.


In [ ]:
# Встроенный комплект v4. Не редактировать.
import base64, hashlib, zipfile, os
from pathlib import Path

_BUNDLE = """..."""
_EXPECTED = "a4fb09838189aa689722ae4655490c068424285904a3f66848670312dd024a8b"

_bundle_path = Path("/content/REINVENT4_MOST_READY_v4.zip")
_bundle_path.write_bytes(base64.b64decode(_BUNDLE))
_actual = hashlib.sha256(_bundle_path.read_bytes()).hexdigest()
assert _actual == _EXPECTED, f"Повреждён встроенный комплект: {_actual}"

ROOT = Path("/content/REINVENT4_MOST_READY")
ROOT.mkdir(parents=True, exist_ok=True)

# ВАЖНО: не удаляем существующий ROOT.
# Так сохраняются уже скачанный upstream, prior и созданные venv из предыдущей попытки.
with zipfile.ZipFile(_bundle_path) as zf:
    zf.extractall("/content")

assert (ROOT / "run.py").is_file()
assert (ROOT / "config_builder.py").is_file()
assert (ROOT / "scoring/models/model_A_wl.joblib").is_file()
assert not list((ROOT / "scoring/models").glob("oracle*.joblib"))

# Проверяем сам конкретный фикс.
_config_text = (ROOT / "config_builder.py").read_text(encoding="utf-8")
assert "unique_sequences = true" not in _config_text

os.chdir(ROOT)
print("Комплект v4 готов:", ROOT)
print("SHA256:", _actual)


In [ ]:
# ================================================================
# ЕДИНСТВЕННЫЙ ЗАПУСК
# ================================================================

SMOKE_STEPS = 3
SMOKE_BATCH = 16
TRAIN_STEPS = 100
BATCH_SIZE = 64
N_MOLECULES = 1000
SEED = 42

import os
import sys
import glob
import json
import time
import shutil
import zipfile
import subprocess
import urllib.request
import tomllib
import hashlib
from pathlib import Path

import pandas as pd
from IPython.display import display

os.chdir(ROOT)
LOG_DIR = ROOT / "colab_logs"
LOG_DIR.mkdir(exist_ok=True)

def banner(msg):
    print("\n" + "=" * 84)
    print(msg)
    print("=" * 84)

def run_live(args, log_name, cwd=ROOT):
    args = [str(x) for x in args]
    log_path = LOG_DIR / log_name
    print("\n>>>", " ".join(args))
    print("Лог:", log_path)
    tail = []

    with log_path.open("w", encoding="utf-8") as log:
        p = subprocess.Popen(
            args,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"},
        )
        for line in p.stdout:
            print(line, end="")
            log.write(line)
            tail.append(line.rstrip())
            if len(tail) > 120:
                tail.pop(0)
        rc = p.wait()

    if rc:
        banner("ОШИБКА — последние строки")
        print("\n".join(tail))
        raise RuntimeError(f"Код {rc}. Полный лог: {log_path}")

    return log_path

def download_retry(url, target, attempts=3):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.is_file() and target.stat().st_size > 1024:
        print("Используем уже скачанный:", target)
        return target

    last = None
    for n in range(1, attempts + 1):
        temp = target.with_suffix(target.suffix + ".partial")
        temp.unlink(missing_ok=True)
        try:
            print(f"Скачивание {n}/{attempts}: {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=180) as src, temp.open("wb") as dst:
                shutil.copyfileobj(src, dst, 1024 * 1024)
            if temp.stat().st_size <= 1024:
                raise RuntimeError("Скачанный файл слишком мал")
            temp.replace(target)
            return target
        except Exception as e:
            last = e
            temp.unlink(missing_ok=True)
            if n < attempts:
                time.sleep(3 * n)

    raise RuntimeError(f"Не удалось скачать {url}: {last}")


# ---------------------------------------------------------------
# Official REINVENT prior from GitHub history. No Zenodo.
PRIOR_COMMIT = "b44124489e8ddeabaccf5856b5d823f1574db063"
PRIOR_BLOB_SHA = "0dee328238b3d413b5a34c80e3ddab49e4e0af28"
PRIOR_SIZE = 23226277
PRIOR_RAW_URL = (
    "https://raw.githubusercontent.com/MolecularAI/REINVENT4/"
    + PRIOR_COMMIT
    + "/priors/reinvent.prior"
)

def git_blob_sha(path):
    path = Path(path)
    data = path.read_bytes()
    h = hashlib.sha1()
    h.update(f"blob {len(data)}\0".encode("ascii"))
    h.update(data)
    return h.hexdigest()

def prior_is_valid(path):
    path = Path(path)
    return (
        path.is_file()
        and path.stat().st_size == PRIOR_SIZE
        and git_blob_sha(path) == PRIOR_BLOB_SHA
    )

def ensure_reinvent_prior(target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)

    for candidate in (ROOT / "priors" / "reinvent.prior", target):
        if prior_is_valid(candidate):
            if candidate != target:
                shutil.copy2(candidate, target)
            print("Prior verified:", candidate)
            print("Git blob SHA:", PRIOR_BLOB_SHA)
            return target

    target.unlink(missing_ok=True)

    try:
        download_retry(PRIOR_RAW_URL, target, attempts=3)
        if prior_is_valid(target):
            print("Prior downloaded from official GitHub history.")
            print("Git blob SHA:", PRIOR_BLOB_SHA)
            return target
        print("Raw GitHub prior failed integrity check; using git fallback.")
        target.unlink(missing_ok=True)
    except Exception as exc:
        print("Raw GitHub download unavailable:", repr(exc))
        target.unlink(missing_ok=True)

    repo = Path("/content/_reinvent_prior_git")
    if repo.exists():
        shutil.rmtree(repo)
    repo.mkdir(parents=True, exist_ok=True)

    subprocess.run(["git", "init", "-q", str(repo)], check=True)
    subprocess.run(
        ["git", "-C", str(repo), "remote", "add", "origin",
         "https://github.com/MolecularAI/REINVENT4.git"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo), "fetch", "-q", "--depth=1", "origin", PRIOR_COMMIT],
        check=True,
    )

    with target.open("wb") as out:
        subprocess.run(
            ["git", "-C", str(repo), "show", "FETCH_HEAD:priors/reinvent.prior"],
            stdout=out,
            check=True,
        )

    if not prior_is_valid(target):
        got_size = target.stat().st_size if target.exists() else -1
        got_sha = git_blob_sha(target) if target.exists() else "missing"
        raise RuntimeError(
            "Official GitHub prior failed integrity validation: "
            f"size={got_size}, blob_sha={got_sha}"
        )

    print("Prior extracted from official REINVENT4 Git history.")
    print("Git blob SHA:", PRIOR_BLOB_SHA)
    return target

# ---------------------------------------------------------------
banner("1/7  GPU и Python 3.12")

GPU = shutil.which("nvidia-smi") is not None
PROCESSOR = "cu126" if GPU else "cpu"
DEVICE = "cuda:0" if GPU else "cpu"

print("GPU:", GPU)
print("PROCESSOR:", PROCESSOR)
print("DEVICE:", DEVICE)
if GPU:
    subprocess.run(["nvidia-smi"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("uv не найден")

subprocess.run([UV, "python", "install", "3.12"], check=True)
PY312 = subprocess.run(
    [UV, "python", "find", "3.12"],
    check=True, text=True, capture_output=True
).stdout.strip()

ENGINE_PY = ROOT / ".venv-engine/bin/python"
EVAL_PY = ROOT / ".venv-evaluators/bin/python"

# ---------------------------------------------------------------
banner("2/7  Среда")

def existing_runtime_ok():
    if not (ENGINE_PY.is_file() and EVAL_PY.is_file()):
        return False
    if not (ROOT / "vendor/REINVENT4/pyproject.toml").is_file():
        return False
    if not prior_is_valid(ROOT / "priors/reinvent.prior"):
        return False

    engine = subprocess.run(
        [
            str(ENGINE_PY), "-c",
            (
                "import scipy, torch, reinvent; "
                "from scipy.stats import gaussian_kde; "
                "from reinvent.Reinvent import main_script; "
                "print(scipy.__version__, torch.__version__, torch.cuda.is_available())"
            ),
        ],
        text=True, capture_output=True,
    )
    evaluator = subprocess.run(
        [
            str(EVAL_PY), "-c",
            (
                "import sklearn,xgboost,rdkit,numpy,joblib; "
                "assert sklearn.__version__=='1.9.1'; "
                "assert xgboost.__version__=='3.4.1'; "
                "print('evaluator OK')"
            ),
        ],
        text=True, capture_output=True,
    )
    if engine.returncode == 0 and evaluator.returncode == 0:
        print("Переиспользуем уже установленную среду.")
        print(engine.stdout.strip())
        print(evaluator.stdout.strip())
        return True
    return False

if not existing_runtime_ok():
    print("Готовой среды нет — выполняем setup.")

    upstream = json.loads((ROOT / "UPSTREAM.json").read_text(encoding="utf-8"))
    cache = ROOT / "colab_cache"
    cache.mkdir(exist_ok=True)

    source_zip = download_retry(
        upstream["source_archive_url"],
        cache / "REINVENT4-pinned.zip",
    )
    prior_file = ensure_reinvent_prior(
        cache / "reinvent.prior",
    )

    for envdir in (ROOT / ".venv-engine", ROOT / ".venv-evaluators"):
        if envdir.exists():
            shutil.rmtree(envdir)

    run_live(
        [
            PY312, "run.py", "setup",
            "--processor", PROCESSOR,
            "--source-zip", source_zip,
            "--prior-file", prior_file,
        ],
        "setup.log",
    )

# Страховка для upstream plot.py.
subprocess.run(
    [str(ENGINE_PY), "-m", "pip", "install", "-q", "scipy==1.18.1"],
    check=True,
)

# Final hard check: exact official prior.
if not prior_is_valid(ROOT / "priors/reinvent.prior"):
    cache = ROOT / "colab_cache"
    cache.mkdir(exist_ok=True)
    verified_prior = ensure_reinvent_prior(cache / "reinvent.prior")
    (ROOT / "priors").mkdir(exist_ok=True)
    shutil.copy2(verified_prior, ROOT / "priors/reinvent.prior")

assert prior_is_valid(ROOT / "priors/reinvent.prior")
print("Official REINVENT prior integrity: OK")

run_live(
    [
        ENGINE_PY, "-c",
        (
            "import scipy,torch,reinvent; "
            "from scipy.stats import gaussian_kde; "
            "from reinvent.Reinvent import main_script; "
            "print('SciPy',scipy.__version__); "
            "print('Torch',torch.__version__); "
            "print('CUDA',torch.cuda.is_available()); "
            "print('REINVENT import OK')"
        ),
    ],
    "runtime.log",
)

# ---------------------------------------------------------------
banner("3/7  Полный check")

run_live(
    [PY312, "run.py", "check", "--seed", str(SEED)],
    "check.log",
)

# ---------------------------------------------------------------
banner("4/7  Валидация RL-конфига ДО обучения")

# Генерируем точно тот же тип конфига, который использует run.py train.
from config_builder import rl_config

preflight_dir = ROOT / "runs" / "_preflight_v5"
preflight_dir.mkdir(parents=True, exist_ok=True)
preflight_toml = preflight_dir / "train.toml"

preflight_toml.write_text(
    rl_config(
        ROOT / "priors/reinvent.prior",
        ROOT / "priors/reinvent.prior",
        preflight_dir,
        EVAL_PY,
        ROOT / "scoring/most_score.py",
        steps=SMOKE_STEPS,
        batch_size=SMOKE_BATCH,
        device=DEVICE,
        resume=False,
    ),
    encoding="utf-8",
)

parsed = tomllib.loads(preflight_toml.read_text(encoding="utf-8"))
assert "unique_sequences" not in parsed["parameters"]

validation_script = r"""
import sys, tomllib
from reinvent.runmodes.RL.validation import RLConfig
path = sys.argv[1]
with open(path, "rb") as f:
    cfg = tomllib.load(f)
allowed = set(RLConfig.model_fields)
payload = {k:v for k,v in cfg.items() if k in allowed}
RLConfig(**payload)
print("RLConfig VALID")
print("parameter keys:", sorted(payload["parameters"]))
"""

run_live(
    [ENGINE_PY, "-c", validation_script, preflight_toml],
    "rl_schema_preflight.log",
)

# ---------------------------------------------------------------
banner("5/7  Smoke-test RL: 3 шага")

run_live(
    [
        PY312, "run.py", "train",
        "--steps", str(SMOKE_STEPS),
        "--batch-size", str(SMOKE_BATCH),
        "--device", DEVICE,
        "--seed", str(SEED),
    ],
    "smoke_train.log",
)

# ---------------------------------------------------------------
banner("6/7  Основной RL")

run_live(
    [
        PY312, "run.py", "train",
        "--steps", str(TRAIN_STEPS),
        "--batch-size", str(BATCH_SIZE),
        "--device", DEVICE,
        "--seed", str(SEED),
    ],
    "full_train.log",
)

# ---------------------------------------------------------------
banner("7/7  Sampling 1000 + результаты")

run_live(
    [
        PY312, "run.py", "sample",
        "--n", str(N_MOLECULES),
        "--device", DEVICE,
        "--seed", str(SEED),
    ],
    "sample.log",
)

property_files = sorted(
    glob.glob(str(ROOT / "runs/sample-*/properties.csv")),
    key=os.path.getmtime,
)
if not property_files:
    raise RuntimeError("Sampling завершился, но properties.csv не найден")

result_file = Path(property_files[-1])
df = pd.read_csv(result_file)

print("Результат:", result_file)
print("Строк:", len(df))

if "joint_pass" in df.columns:
    c = df["joint_pass"]
    if c.dtype == object:
        mask = c.astype(str).str.lower().isin(["true", "1", "yes"])
    else:
        mask = c.fillna(False).astype(bool)

    good = df.loc[mask].copy()
    if "joint" in good.columns:
        good = good.sort_values("joint", ascending=False)

    good.to_csv(ROOT / "best_candidates.csv", index=False)

    print("joint_pass:", len(good))
    if len(df):
        print(f"Joint success rate: {100*len(good)/len(df):.2f}%")
    display(good.head(30) if len(good) else df.head(20))
else:
    display(df.head(20))

# Собираем один результатный ZIP.
outzip = Path("/content/REINVENT4_MOST_RESULTS.zip")
outzip.unlink(missing_ok=True)

with zipfile.ZipFile(outzip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in (ROOT / "runs").rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(ROOT))
    best = ROOT / "best_candidates.csv"
    if best.is_file():
        z.write(best, best.relative_to(ROOT))
    for p in LOG_DIR.glob("*.log"):
        z.write(p, p.relative_to(ROOT))

banner("ГОТОВО")
print("Итоговый файл:", outzip)
print("Скачайте через Colab → Files.")


После завершения заберите:

`/content/REINVENT4_MOST_RESULTS.zip`
